<a href="https://colab.research.google.com/github/ms914/3DKWM_server/blob/main/kwm_cli/KWM_CLI_Mehrfachbindungen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://share.gemini.google/vVXC6mxD25Cp

## Backend

In [1]:
from fastapi import FastAPI
from scipy.spatial.transform import Rotation as R
import numpy as np
import threading
import uvicorn
import time

app = FastAPI()

# Globaler Zustand des Moleküls
if "WELT_ZUSTAND" not in globals():
    WELT_ZUSTAND = {}

# Physikalische Bindungsabstände in Angström
ELEMENT_PROFIL = {
    "C": {"abstand": 1.54, "doppel": 1.34, "dreifach": 1.20},
    "H": {"abstand": 1.09},
    "O": {"abstand": 1.43, "doppel": 1.21} # O=O oder C=O Bindungslänge
}

def berechne_ausrichtungs_matrix(start_vektor, ziel_vektor):
    """
    Berechnet die Rotationsmatrix, die den 'start_vektor' exakt in den 'ziel_vektor' überführt.
    Nutzt die mathematisch robuste algebraische Methode über das Kreuzprodukt.
    """
    # Vektoren normieren
    v_from = start_vektor / np.linalg.norm(start_vektor)
    v_to = ziel_vektor / np.linalg.norm(ziel_vektor)

    # Wenn Vektoren bereits identisch sind, keine Drehung
    if np.allclose(v_from, v_to):
        return np.eye(3)
    # Wenn Vektoren exakt entgegengesetzt sind, 180-Grad-Drehung um eine orthogonale Achse
    if np.allclose(v_from, -v_to):
        # Finde eine orthogonale Achse
        ortho = np.array([1.0, 0.0, 0.0]) if not np.allclose(abs(v_from[0]), 1.0) else np.array([0.0, 1.0, 0.0])
        achse = np.cross(v_from, ortho)
        achse /= np.linalg.norm(achse)
        return R.from_rotvec(achse * np.pi).as_matrix()

    # Standard-Rotationsberechnung über Kreuz- und Skalarprodukt
    v = np.cross(v_from, v_to)
    c = np.dot(v_from, v_to)
    s = np.linalg.norm(v)
    k_matrix = np.array([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])
    rotations_matrix = np.eye(3) + k_matrix + np.dot(k_matrix, k_matrix) * ((1 - c) / (s ** 2))
    return rotations_matrix

@app.get("/status")
def get_status():
    return WELT_ZUSTAND

@app.post("/erzeuge")
def erzeuge(daten: dict):
    symbol = daten.get("symbol", "C").upper()
    uid = f"{symbol}_{len(WELT_ZUSTAND) + 1}"

    # Standard-Tetraeder-Vektoren für die Geometrie
    tetraeder_vektoren = [
        [0.0, 1.0, -1.0 / np.sqrt(2)],
        [0.0, -1.0, -1.0 / np.sqrt(2)],
        [1.0, 0.0, 1.0 / np.sqrt(2)],
        [-1.0, 0.0, 1.0 / np.sqrt(2)]
    ]
    wolken_lokal = [(v / np.linalg.norm(v)).tolist() for v in tetraeder_vektoren]

    # Element-spezifische Lewis-Wolken initialisieren
    if symbol == "H":
        # Wasserstoff hat nur 1 Valenzelektron
        wolken = [
            {"lokal": wolken_lokal[0], "elektronen": 1, "partner": None}
        ]
    elif symbol == "O":
        # Sauerstoff hat 6 Valenzelektronen: 2 Paare (2) und 2 Radikale (1)
        wolken = [
            {"lokal": wolken_lokal[0], "elektronen": 2, "partner": None}, # Freies Paar |
            {"lokal": wolken_lokal[1], "elektronen": 2, "partner": None}, # Freies Paar |
            {"lokal": wolken_lokal[2], "elektronen": 1, "partner": None}, # Radikal •
            {"lokal": wolken_lokal[3], "elektronen": 1, "partner": None}  # Radikal •
        ]
    else: # Standard: Kohlenstoff (C)
        wolken = [
            {"lokal": wolken_lokal[0], "elektronen": 1, "partner": None},
            {"lokal": wolken_lokal[1], "elektronen": 1, "partner": None},
            {"lokal": wolken_lokal[2], "elektronen": 1, "partner": None},
            {"lokal": wolken_lokal[3], "elektronen": 1, "partner": None}
        ]

    WELT_ZUSTAND[uid] = {
        "symbol": symbol,
        "position": [float(np.random.uniform(-1, 1)), float(np.random.uniform(-1, 1)), float(np.random.uniform(-1, 1))],
        "rotor_bivektor": [0.0, 0.0, 0.0],
        "wolken": wolken
    }
    return {"status": "success", "uid": uid}

@app.post("/binde")
def binde(daten: dict):
    try:
        if daten["atom_A_id"] not in WELT_ZUSTAND or daten["atom_B_id"] not in WELT_ZUSTAND:
            return {"status": "error", "message": "Eines der Atome existiert nicht."}

        atom_A = WELT_ZUSTAND[daten["atom_A_id"]]
        atom_B = WELT_ZUSTAND[daten["atom_B_id"]]

        for w in atom_A["wolken"]: w.setdefault("partner", None)
        for w in atom_B["wolken"]: w.setdefault("partner", None)

        # 1. Bindungsordnung bestimmen
        bereits_gebunden_A = [w for w in atom_A["wolken"] if w["partner"] == daten["atom_B_id"]]
        bereits_gebunden_B = [w for w in atom_B["wolken"] if w["partner"] == daten["atom_A_id"]]
        ziel_ordnung = len(bereits_gebunden_A) + 1

        if ziel_ordnung > 3:
            return {"status": "error", "message": "Maximale Bindungsordnung erreicht."}



        # 2. Freie Wolke finden (Priorisiere Radikale mit 1 Elektron vor freien Paaren)
        # Suche erst nach echten Radikalen (1 Elektron)
        aktuelle_wolke_A = next((w for w in atom_A["wolken"] if w["elektronen"] == 1 and w["partner"] != daten["atom_B_id"]), None)
        # Falls kein Radikal da ist, nimm eine Wolke mit 2 Elektronen (koordinative Bindung/freies Paar)
        if not aktuelle_wolke_A:
            aktuelle_wolke_A = next((w for w in atom_A["wolken"] if w["elektronen"] == 2 and w["partner"] != daten["atom_B_id"]), None)

        # Das gleiche Spiel für Atom B
        aktuelle_wolke_B = next((w for w in atom_B["wolken"] if w["elektronen"] == 1 and w["partner"] != daten["atom_A_id"]), None)
        if not aktuelle_wolke_B:
            aktuelle_wolke_B = next((w for w in atom_B["wolken"] if w["elektronen"] == 2 and w["partner"] != daten["atom_A_id"]), None)

        if not aktuelle_wolke_A or not aktuelle_wolke_B:
            return {"status": "error", "message": "Keine freien Wolken verfügbar."}


        # 3. Abstandslogik bestimmen (Dynamisch nach Bindungsordnung)
        element_paar = sorted([atom_A["symbol"], atom_B["symbol"]])

        if element_paar == ["C", "C"]:
            if ziel_ordnung == 1: abstand = ELEMENT_PROFIL["C"]["abstand"]
            elif ziel_ordnung == 2: abstand = ELEMENT_PROFIL["C"]["doppel"]
            else: abstand = ELEMENT_PROFIL["C"]["dreifach"]

        elif element_paar == ["C", "O"] or element_paar == ["O", "O"]:
            if ziel_ordnung == 1: abstand = ELEMENT_PROFIL["O"]["abstand"]
            else: abstand = ELEMENT_PROFIL["O"]["doppel"]

        else:
            # Standard/Wasserstoff-Bindungen (H hat keine Mehrfachbindungen)
            abstand = ELEMENT_PROFIL["H"]["abstand"] if "H" in element_paar else ELEMENT_PROFIL["C"]["abstand"]

        # 4. Geometrische Achse (Mittelwert über Altdaten + neue Wunschwolke)
        beteiligte_wolken_A = bereits_gebunden_A + [aktuelle_wolke_A]
        lokaler_vektor_A = np.sum([np.array(w["lokal"]) for w in beteiligte_wolken_A], axis=0)
        achse_A = lokaler_vektor_A / np.linalg.norm(lokaler_vektor_A) if np.linalg.norm(lokaler_vektor_A) != 0 else np.array([1.0, 0.0, 0.0])

        # Atom B im Raum verschieben (Echte Python-Floats!)
        neue_pos_B = np.array(atom_A["position"]) + (achse_A * abstand)
        atom_B["position"] = [float(x) for x in neue_pos_B]

        # 5. ROTATIONS-AUSRICHTUNG (Die SciPy/NumPy Matrix-Kaskade)
        ziel_achse_B = -achse_A
        beteiligte_wolken_B = bereits_gebunden_B + [aktuelle_wolke_B]
        start_vektor_B = np.sum([np.array(w["lokal"]) for w in beteiligte_wolken_B], axis=0)
        start_achse_B = start_vektor_B / np.linalg.norm(start_vektor_B) if np.linalg.norm(start_vektor_B) != 0 else np.array([1.0, 0.0, 0.0])

        # Basis-Ausrichtung berechnen
        R_matrix = berechne_ausrichtungs_matrix(start_achse_B, ziel_achse_B)

        # Axiale Torsion anwenden (Doppelbindung konform auf 0°, Dreifachbindung auf 60°)
        if ziel_ordnung == 2:
            R_torsion = R.from_rotvec(achse_A * np.radians(0)).as_matrix()
            R_matrix = np.dot(R_torsion, R_matrix)
        elif ziel_ordnung == 3:
            R_torsion = R.from_rotvec(achse_A * np.radians(60)).as_matrix()
            R_matrix = np.dot(R_torsion, R_matrix)


        # Ermittle die aktuelle Bindungsordnung, die GERADE GEKNÜPFT WIRD (1=Einfach, 2=Doppel)
        ziel_ordnung = sum(1 for wl in atom_A["wolken"] if wl.get("partner") == daten["atom_B_id"]) + 1

        if ziel_ordnung == 2:
            # Prüfen, ob das Zentralatom (z.B. Kohlenstoff) bereits eine ANDERE Doppelbindung hat
            bereits_doppelbindung = False
            for w in atom_A["wolken"]:
                if w.get("partner") and w["partner"] != daten["atom_B_id"]:
                    andere_ordnung = sum(1 for wl in atom_A["wolken"] if wl.get("partner") == w["partner"])
                    if andere_ordnung >= 2:
                        bereits_doppelbindung = True
                        break

            if bereits_doppelbindung:
                # Zweite Doppelbindung: 90 Grad orthogonal + Korrekturfaktor
                # (Probiere 90 + 35 oder 90 - 35, je nachdem in welche Richtung dein System dreht)
                R_torsion = R.from_rotvec(achse_A * np.radians(0)).as_matrix()
            else:
                # Erste Doppelbindung: Wir drehen um exakt 35.26 Grad,
                # um die gestaffelte Tetraeder-Ausrichtung auf 'ekliptisch' (Spitze-auf-Spitze) zu zwingen!
                R_torsion = R.from_rotvec(achse_A * np.radians(35.26)).as_matrix()

            R_matrix = np.dot(R_torsion, R_matrix)

        # Konvertiere Rotationsmatrix in SciPy-Objekt zur Extraktion des Pseudo-Bivektors/Rotationsvektors
        scipy_rot = R.from_matrix(R_matrix)
        rot_vektor = scipy_rot.as_rotvec() # Liefert [x, y, z] Drehvektor (Isomorph zu CGA-Bivektorkomponenten)
        atom_B["rotor_bivektor"] = [float(x) for x in rot_vektor]

        # Wende die Rotation lokal auf alle Wolken von Atom B an, damit das molekulare Gefüge stimmt
        for w in atom_B["wolken"]:
            w_lokal_neu = np.dot(R_matrix, np.array(w["lokal"]))
            w["lokal"] = [float(x) for x in w_lokal_neu]

        # JSON-Schutz für Wolken von Atom A (Sicherstellen, dass alles primitive Typen sind)
        for w in atom_A["wolken"]:
            w["lokal"] = [float(x) for x in w["lokal"]]

        # 6. Status der Elektronenwolken updaten
        aktuelle_wolke_A["elektronen"] = 3
        aktuelle_wolke_A["partner"] = daten["atom_B_id"]
        aktuelle_wolke_B["elektronen"] = 3
        aktuelle_wolke_B["partner"] = daten["atom_A_id"]

        return {"status": "success", "bindung_aktuell": ziel_ordnung}

    except Exception as e:
        return {"status": "error", "message": f"Fehler in der NumPy/SciPy Engine: {str(e)}"}

# --- ASYNCHRONER THREAD-START (Verhindert Colab-Blockade) ---
def start_server():
    import nest_asyncio
    nest_asyncio.apply()
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

time.sleep(1.5)
print("🚀 Produktionsreifer NumPy/SciPy-Molekülserver läuft sicher im Hintergrund auf http://127.0.0.1:8000 !")

🚀 Produktionsreifer NumPy/SciPy-Molekülserver läuft sicher im Hintergrund auf http://127.0.0.1:8000 !


## Frontend

In [2]:
! pip install rdkit -q
import requests
import json
from rdkit import Chem
import copy
import sys

# Speichert Schnappschüsse des gesamten WELT_ZUSTANDs nach jedem Einzelschritt
SMILES_HISTORY = []

def generiere_mol_lokal(welt_zustand):
    # Generiert das MDL MOL-File direkt im Frontend aus den Statusdaten
    if not welt_zustand:
        return ""

    mol_text = "CGA_Baukasten_Export\n  CGA-Engine 2026\n\n"
    anzahl_atome = len(welt_zustand)

    bindungen = []
    gesehene_paare = set()
    uid_zu_index = {uid: idx + 1 for idx, uid in enumerate(welt_zustand.keys())}

    for uid_A, atom in welt_zustand.items():
        for w in atom.get('wolken', []):
            uid_B = w.get('partner')
            if uid_B and uid_B in welt_zustand:
                paar = tuple(sorted([uid_A, uid_B]))
                if paar not in gesehene_paare:
                    gesehene_paare.add(paar)
                    ordnung = sum(1 for wl in atom['wolken'] if wl.get('partner') == uid_B)
                    bindungen.append((uid_zu_index[uid_A], uid_zu_index[uid_B], ordnung))

    anzahl_bindungen = len(bindungen)
    mol_text += "%3d%3d  0  0  0  0  0  0  0  0999 V2000\n" % (anzahl_atome, anzahl_bindungen)

    for uid, atom in welt_zustand.items():
        pos = atom['position']
        symbol = atom['symbol']
        mol_text += "%10.4f%10.4f%10.4f %-3s 0  0  0  0  0  0  0  0  0  0  0  0\n" % (pos[0], pos[1], pos[2], symbol)

    for b in bindungen:
        mol_text += "%3d%3d%3d  0  0  0  0\n" % (b[0], b[1], b[2])

    mol_text += "M  END\n"
    return mol_text

SMILES_HISTORIE = []  # nur der String

def lade_smiles_in_backend(smiles_string):

    global SMILES_HISTORY # die Welt_Zustände als snapshots
    SMILES_HISTORY = []  # Historie für das neue Molekül zurücksetzen

    # Parst einen SMILES-String mit RDKit und sendet die Befehle automatisiert an das Backend
    mol = Chem.MolFromSmiles(smiles_string)
    if mol is None:
        print(f"-> Fehler: Ungueltiger SMILES-String '{smiles_string}'")
        return False

    # Wasserstoff-Atome explizit hinzufügen
    mol = Chem.AddHs(mol)

    print(f"-> Interpretiere SMILES: {smiles_string}")
    print(f"-> Gefunden: {mol.GetNumAtoms()} Atome (inkl. H) und {mol.GetNumBonds()} Bindungen.")

    rdkit_idx_zu_uid = {}

    # 1. Schritt: Alle Atome im Backend erzeugen
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        rdkit_idx = atom.GetIdx()

        if symbol not in ["C", "H", "O"]:
            print(f"-> Warnung: Element {symbol} wird aktuell nicht unterstuetzt. Ueberspringe.")
            continue

        try:
            res = requests.post("http://127.0.0.1:8000/erzeuge", json={"symbol": symbol}).json()
            if res.get("status") == "success":
                rdkit_idx_zu_uid[rdkit_idx] = res['uid']
                # --- SCHNAPPSCHUSS NACH ATOM-ERZEUGUNG ---
                aktuell = requests.get("http://127.0.0.1:8000/status").json()
                SMILES_HISTORY.append(copy.deepcopy(aktuell))
            else:
                print(f"-> Fehler beim Erzeugen von {symbol}: {res.get('message')}")
        except:
            print("-> Fehler: Server nicht erreichbar.")
            return False

    # 2. Schritt: Alle Bindungen im Backend knuepfen
    for bond in mol.GetBonds():
        idx_A = bond.GetBeginAtomIdx()
        idx_B = bond.GetEndAtomIdx()

        if idx_A in rdkit_idx_zu_uid and idx_B in rdkit_idx_zu_uid:
            uid_A = rdkit_idx_zu_uid[idx_A]
            uid_B = rdkit_idx_zu_uid[idx_B]

            b_type = bond.GetBondType()
            wiederholungen = 1
            if b_type == Chem.BondType.DOUBLE:
                wiederholungen = 2
            elif b_type == Chem.BondType.TRIPLE:
                wiederholungen = 3

            for _ in range(wiederholungen):
                try:
                    requests.post("http://127.0.0.1:8000/binde", json={"atom_A_id": uid_A, "atom_B_id": uid_B})
                    # --- SCHNAPPSCHUSS NACH JEDER BINDUNG (auch jede Teildoppelbindung!) ---
                    aktuell = requests.get("http://127.0.0.1:8000/status").json()
                    SMILES_HISTORY.append(copy.deepcopy(aktuell))
                except:
                    print("-> Fehler beim Knuepfen einer Bindung (Server-Verbindung verloren).")
                    return False

    print(f"-> SMILES erfolgreich geladen! {len(SMILES_HISTORY)} Einzelschritte aufgezeichnet.")
    # --- HIER WIRD DIE HISTORIE BEFÜLLT ---
    if smiles_string not in SMILES_HISTORIE:
        SMILES_HISTORIE.append(smiles_string)
    return True


def zeige_welt(welt_zustand):
    print("\n" + "="*95)
    print(" MOLEKÜL-BAUKASTEN (LEWIS-SCHREIBWEISE)")
    print("="*95)

    if not welt_zustand:
        print(" Keine Atome in der Szene. Nutze 'erzeuge <C/H/O>' oder 'smiles <string>' um zu starten.")
        print("="*95)
        return

    for uid, atom in welt_zustand.items():
        symbol = atom["symbol"]
        pos = atom["position"]
        pos_str = [f"{p:.2f}" for p in pos]

        kwm_symbole = []
        freie_paare = 0
        radikale = 0

        # 1. KWM-Feld und Zählung der ungebundenen Zustände präzisieren
        for w in atom["wolken"]:
            if w.get("partner"):
                p_id = w["partner"]
                bindungsstufe = sum(1 for wl in atom["wolken"] if wl.get("partner") == p_id)
                if bindungsstufe == 3: kwm_symbole.append("≡")
                elif bindungsstufe == 2: kwm_symbole.append("═")
                else: kwm_symbole.append("─")
            else:
                # Unterscheidung: Volle Wolke = freies Paar (|), Halbe Wolke = Radikal (•)
                if w.get("elektronen", 1) == 2:
                    kwm_symbole.append("|")
                    freie_paare += 1
                else:
                    kwm_symbole.append("•")
                    radikale += 1

        kwm_str = " ".join(kwm_symbole)

        # 2. Lewis-Feld semantisch korrekt aufbauen
        # Freie Elektronenpaare umschließen das Elementsymbol (z.B. |O| )
        lewis_str = "|" * freie_paare + symbol + "•" * radikale

        partner_erwaehnt = set()
        for w in atom["wolken"]:
            p_id = w.get("partner")
            if p_id and p_id not in partner_erwaehnt:
                partner_erwaehnt.add(p_id)
                bindungsstufe = sum(1 for wl in atom["wolken"] if wl.get("partner") == p_id)
                if bindungsstufe == 3: lewis_str += f" ≡{p_id}"
                elif bindungsstufe == 2: lewis_str += f" ═{p_id}"
                else: lewis_str += f" ─{p_id}"

        print(f"ID: {uid:<6} | Pos: {pos_str} | KWM: [ {kwm_str:<7} ] | Lewis: {lewis_str}")
    print("="*95)


def zeige_smiles_historie():
    print("\n" + "-"*40)
    print(" LADE-HISTORIE (SMILES)")
    print("-"*40)
    if not SMILES_HISTORIE:
        print(" Noch keine SMILES-Strings geladen.")
    else:
        for idx, smiles in enumerate(SMILES_HISTORIE, 1):
            print(f" [{idx}] {smiles}")
    print("-"*40 + "\n")

import requests

def starte_baukasten_mit_smiles(smiles_code, export_dateiname="molekuel.mol"):
    """
    Verarbeitet einen SMILES-Code vollautomatisch in Colab,
    exportiert die Geometrie und beendet den Vorgang.
    """
    print(f"Colab-Autopilot aktiv. Verarbeite SMILES: {smiles_code}")

    # 1. SMILES parsen und an das Backend senden
    erfolg = lade_smiles_in_backend(smiles_code)

    if erfolg:
        try:
            # 2. Den berechneten Zustand aus dem Backend abholen
            res = requests.get("http://127.0.0.1:8000/status")
            aktuelle_welt = res.json()

            # 3. MOL-Struktur generieren und in Colab speichern
            mol_data = generiere_mol_lokal(aktuelle_welt)
            with open(export_dateiname, "w") as f:
                f.write(mol_data)

            print(f"-> Erfolg! '{export_dateiname}' wurde in deiner Colab-Umgebung gespeichert.")

        except Exception as e:
            print(f"-> Fehler beim automatischen Export: {e}")
    else:
        print("-> Automatische Verarbeitung abgebrochen aufgrund eines Fehlers beim Parsen.")

    print("Vorgang abgeschlossen.")

def cli_baukasten():
    global WELT_ZUSTAND


    # --- RELEVANT FÜR DEN NORMALEN / INTERAKTIVEN MODUS ---
    print("CLI-Frontend erfolgreich geladen.")
    aktuelle_welt = {}

    try:
        res = requests.get("http://127.0.0.1:8000/status")
        aktuelle_welt = res.json()
        zeige_welt(aktuelle_welt)
    except:
        zeige_welt({})

    while True:
        eingabe = input("Befehl [erzeuge <S> | binde <id1> <id2> | smiles <string> | history | export | exit]: ").strip().split()
        if not eingabe: continue
        kommando = eingabe[0].lower()

        if kommando == "exit":
            print("Baukasten beendet. Auf Wiedersehen!")
            break

        elif kommando == "history":
            zeige_smiles_historie()

        elif kommando == "smiles" and len(eingabe) > 1:
            smiles_str = eingabe[1]
            lade_smiles_in_backend(smiles_str)

        elif kommando == "erzeuge" and len(eingabe) > 1:
            symbol = eingabe[1].upper()
            if symbol in ["C", "H", "O"]:
                try: requests.post("http://127.0.0.1:8000/erzeuge", json={"symbol": symbol})
                except: print("-> Fehler: Server offline.")
            else: print("-> Ungueltiges Element.")

        elif kommando == "binde" and len(eingabe) > 2:
            try: requests.post("http://127.0.0.1:8000/binde", json={"atom_A_id": eingabe[1], "atom_B_id": eingabe[2]})
            except: print("-> Fehler: Verbindung fehlgeschlagen.")

        elif kommando == "export":
            if aktuelle_welt:
                mol_data = generiere_mol_lokal(aktuelle_welt)
                with open("molekuel.mol", "w") as f:
                    f.write(mol_data)
                print("-> GENIAL! 'molekuel.mol' wurde direkt im Frontend generiert und gespeichert.")
            else:
                print("-> Fehler: Keine Daten zum Exportieren vorhanden.")
        else:
            print("-> Unbekannter Befehl.")

        try:
            res = requests.get("http://127.0.0.1:8000/status")
            aktuelle_welt = res.json()
            zeige_welt(aktuelle_welt)
        except:
            print(" [Warnung] Synchronisation fehlgeschlagen.")

# CLI direkt ausführen
cli_baukasten()

CLI-Frontend erfolgreich geladen.

 MOLEKÜL-BAUKASTEN (LEWIS-SCHREIBWEISE)
 Keine Atome in der Szene. Nutze 'erzeuge <C/H/O>' oder 'smiles <string>' um zu starten.
Befehl [erzeuge <S> | binde <id1> <id2> | smiles <string> | history | export | exit]: smiles C=C
-> Interpretiere SMILES: C=C
-> Gefunden: 6 Atome (inkl. H) und 5 Bindungen.
-> SMILES erfolgreich geladen! 12 Einzelschritte aufgezeichnet.

 MOLEKÜL-BAUKASTEN (LEWIS-SCHREIBWEISE)
ID: C_1    | Pos: ['0.37', '-0.29', '0.45'] | KWM: [ ═ ═ ─ ─ ] | Lewis: C ═C_2 ─H_3 ─H_4
ID: C_2    | Pos: ['0.37', '-0.29', '-0.89'] | KWM: [ ═ ═ ─ ─ ] | Lewis: C ═C_1 ─H_5 ─H_6
ID: H_3    | Pos: ['1.26', '-0.29', '1.08'] | KWM: [ ─       ] | Lewis: H ─C_1
ID: H_4    | Pos: ['-0.52', '-0.29', '1.08'] | KWM: [ ─       ] | Lewis: H ─C_1
ID: H_5    | Pos: ['-0.35', '0.22', '-1.52'] | KWM: [ ─       ] | Lewis: H ─C_2
ID: H_6    | Pos: ['1.10', '-0.81', '-1.52'] | KWM: [ ─       ] | Lewis: H ─C_2
Befehl [erzeuge <S> | binde <id1> <id2> | smiles <string> |

## Smiles Molekülbau

In [ ]:
# Einfach den gewünschten SMILES-Code hier übergeben
starte_baukasten_mit_smiles("C=C") # Ethen

Colab-Autopilot aktiv. Verarbeite SMILES: C=C
-> Interpretiere SMILES: C=C
-> Gefunden: 6 Atome (inkl. H) und 5 Bindungen.
-> SMILES erfolgreich geladen! 12 Einzelschritte aufgezeichnet.
-> Erfolg! 'molekuel.mol' wurde in deiner Colab-Umgebung gespeichert.
Vorgang abgeschlossen.


In [10]:
starte_baukasten_mit_smiles("C#C") # Ethin

Colab-Autopilot aktiv. Verarbeite SMILES: C#C
-> Interpretiere SMILES: C#C
-> Gefunden: 4 Atome (inkl. H) und 3 Bindungen.
-> SMILES erfolgreich geladen! 9 Einzelschritte aufgezeichnet.
-> Erfolg! 'molekuel.mol' wurde in deiner Colab-Umgebung gespeichert.
Vorgang abgeschlossen.


In [8]:
starte_baukasten_mit_smiles("O=C=O") # Kohlendioxid

Colab-Autopilot aktiv. Verarbeite SMILES: O=C=O
-> Interpretiere SMILES: O=C=O
-> Gefunden: 3 Atome (inkl. H) und 2 Bindungen.
-> SMILES erfolgreich geladen! 7 Einzelschritte aufgezeichnet.
-> Erfolg! 'molekuel.mol' wurde in deiner Colab-Umgebung gespeichert.
Vorgang abgeschlossen.


In [11]:
starte_baukasten_mit_smiles("C") # Methan

Colab-Autopilot aktiv. Verarbeite SMILES: C
-> Interpretiere SMILES: C
-> Gefunden: 5 Atome (inkl. H) und 4 Bindungen.
-> SMILES erfolgreich geladen! 9 Einzelschritte aufgezeichnet.
-> Erfolg! 'molekuel.mol' wurde in deiner Colab-Umgebung gespeichert.
Vorgang abgeschlossen.


In [ ]:
starte_baukasten_mit_smiles("O") # Wasser

In [ ]:
starte_baukasten_mit_smiles("O=O") # Dioxid

In [ ]:
starte_baukasten_mit_smiles("C=O") # Kohlenmonoxid

In [ ]:
# Erzeuge mehrere Mol Files
beispiel_smiles = ["C", "C=C", "CCO"]

for idx, smiles in enumerate(beispiel_smiles):
    dateiname = f"molekuel_{idx}.mol"
    starte_baukasten_mit_smiles(smiles, export_dateiname=dateiname)

## Visualisierung snapshots

In [5]:
!pip install py3Dmol -q
import py3Dmol
import numpy as np
from ipywidgets import interact, IntSlider

def render_schnappschuss(schritt_index):
    """ Rendert exakt einen Zustand aus der aufgezeichneten Historie """
    if not SMILES_HISTORY:
        print("Keine Historie vorhanden. Lade zuerst ein Molekül via 'smiles <string>'.")
        return

    welt_zustand = SMILES_HISTORY[schritt_index]

    view = py3Dmol.view(width=600, height=400)
    view.setBackgroundColor('#232323')

    mol_text = "CGA_Baukasten_History\n\n"
    wolken_daten = []
    uids = list(welt_zustand.keys())
    uid_zu_idx = {uid: idx + 1 for idx, uid in enumerate(uids)}

    atome_block = ""
    bindungs_block = ""
    anzahl_atome = len(uids)
    anzahl_bindungen = 0
    gesehene_bindungen = set()

    for uid, atom in welt_zustand.items():
        pos = atom["position"]
        symbol = atom["symbol"]
        atome_block += "%10.4f%10.4f%10.4f %-3s 0  0  0  0  0  0  0  0  0  0  0  0\n" % (pos[0], pos[1], pos[2], symbol)

        # Wolken (Orbitale) auswerten
        for w in atom.get("wolken", []):
            lokal = np.array(w["lokal"])
            partner = w.get("partner")
            elektronen = w.get("elektronen", 1)

            # Standard-Radius für freie Elektronen/Radikale
            wolken_radius_faktor = 0.52
            typ = "radikal"

            if partner and partner in welt_zustand:
                typ = "bindung"
                # --- HIER IST DIE OPTISCHE ANPASSUNG ---
                # Bei Bindungen schieben wir die Wolke etwas weiter raus (0.60 statt 0.52),
                # damit sie sich im Zentrum der Bindungsachse mit der Partnerwolke überschneidet
                wolken_radius_faktor = 0.60

                paar = tuple(sorted([uid, partner]))
                if paar not in gesehene_bindungen:
                    gesehene_bindungen.add(paar)
                    ordnung = sum(1 for wl in atom['wolken'] if wl.get('partner') == partner)
                    bindungs_block += "%3d%3d%3d  0  0  0  0\n" % (uid_zu_idx[uid], uid_zu_idx[partner], ordnung)
                    anzahl_bindungen += 1
            elif elektronen == 2:
                typ = "paar"

            # Globale Position berechnen
            globale_wolken_pos = np.array(pos) + (lokal * wolken_radius_faktor)
            wolken_daten.append((globale_wolken_pos, typ))

    header_block = "%3d%3d  0  0  0  0  0  0  0  0999 V2000\n" % (anzahl_atome, anzahl_bindungen)
    komplettes_mol = mol_text + header_block + atome_block + bindungs_block + "M  END\n"

    view.addModel(komplettes_mol, "mol")
    view.setStyle({'model': -1}, {
        'stick': {'colorscheme': 'Jmol', 'radius': 0.15},
        'sphere': {'colorscheme': 'Jmol', 'radius': 0.4}
    })

    for pos_w, typ in wolken_daten:
        if typ == "bindung":
            farbe = "#00FFCC"; radius = 0.28
        elif typ == "paar":
            farbe = "#FFCC00"; radius = 0.25
        else:
            farbe = "#FF3333"; radius = 0.20

        view.addSphere({
            'center': {'x': float(pos_w[0]), 'y': float(pos_w[1]), 'z': float(pos_w[2])},
            'radius': radius, 'color': farbe, 'alpha': 0.75
        })

    view.zoomTo()
    print(f"Schritt {schritt_index + 1} von {len(SMILES_HISTORY)}")
    return view.show()

def zeige_smiles_animation():
    """ Erzeugt den interaktiven Schieberegler im Notebook """
    if not SMILES_HISTORY:
        print("Keine Historie vorhanden. Bitte lade zuerst ein Molekül über das CLI (z.B. smiles C=C).")
        return

    interact(
        render_schnappschuss,
        schritt_index=IntSlider(min=0, max=len(SMILES_HISTORY)-1, step=1, value=0, description='Schritt:')
    )

In [6]:
zeige_smiles_animation()

interactive(children=(IntSlider(value=0, description='Schritt:', max=11), Output()), _dom_classes=('widget-int…

In [7]:
!pip install -q py3Dmol
import py3Dmol
import requests
import numpy as np

def visualisiere_kugelwolken_live():
    # 1. Aktuellen Zustand vom Backend abfragen
    try:
        res = requests.get("http://127.0.0.1:8000/status")
        welt_zustand = res.json()
    except Exception:
        print("-> Fehler: Backend-Server nicht erreichbar. Läuft die API?")
        return

    if not welt_zustand:
        print("Keine Atome in der Szene vorhanden.")
        return

    # 2. py3Dmol Viewer initialisieren (Breite: 600px, Höhe: 400px)
    view = py3Dmol.view(width=600, height=400)
    view.setBackgroundColor('#232323') # Dunkler, kontrastreicher Hintergrund

    mol_text = "CGA_Baukasten_Visualizer\n\n"

    # Listen für das manuelle Zeichnen der Kugelwolken-Dummys
    wolken_daten = []

    # Temporäre Zuordnung für Bindungen im XYZ/MOL-Format
    uids = list(welt_zustand.keys())
    uid_zu_idx = {uid: idx + 1 for idx, uid in enumerate(uids)}

    atome_block = ""
    bindungs_block = ""
    anzahl_atome = len(uids)
    anzahl_bindungen = 0
    gesehene_bindungen = set()

    # 3. Haupt-Atome und Wolken-Geometrie parsen
    for uid, atom in welt_zustand.items():
        pos = atom["position"]
        symbol = atom["symbol"]

        # Atomzeile für das Basis-Gerüst schreiben
        atome_block += "%10.4f%10.4f%10.4f %-3s 0  0  0  0  0  0  0  0  0  0  0  0\n" % (pos[0], pos[1], pos[2], symbol)

        # Wolken (Orbitale) auswerten
        for w in atom.get("wolken", []):
            # Berechne globale Position der Wolke: Atom-Zentrum + (Lokaler Vektor * Skalierung)
            # Wolken liegen für die Optik ca. 0.5 Angström vom Kern entfernt
            lokal = np.array(w["lokal"])
            globale_wolken_pos = np.array(pos) + (lokal * 0.52)

            partner = w.get("partner")
            elektronen = w.get("elektronen", 1)

            if partner:
                # Bindende Kugelwolke
                typ = "bindung"
                paar = tuple(sorted([uid, partner]))
                if paar not in gesehene_bindungen:
                    gesehene_bindungen.add(paar)
                    # Bindungs-Ordnung für das Hauptgerüst ermitteln
                    ordnung = sum(1 for wl in atom['wolken'] if wl.get('partner') == partner)
                    bindungs_block += "%3d%3d%3d  0  0  0  0\n" % (uid_zu_idx[uid], uid_zu_idx[partner], ordnung)
                    anzahl_bindungen += 1

            # === HIER IST DIE KORREKTUR: Chemische Typerkennung ===
            #elif symbol == "O":
                # Bei Sauerstoff sind alle ungebundenen Wolken im stabilen Zustand IMMER freie Paare!
            #    typ = "paar"


            elif elektronen == 2:
                # Freies Elektronenpaar
                typ = "paar"
            else:
                # Ungepaartes Radikal-Elektron
                typ = "radikal"


            wolken_daten.append((globale_wolken_pos, typ))

    # Zusammenbau des MOL-Files für die Haupt-Atome
    header_block = "%3d%3d  0  0  0  0  0  0  0  0999 V2000\n" % (anzahl_atome, anzahl_bindungen)
    komplettes_mol = mol_text + header_block + atome_block + bindungs_block + "M  END\n"

    # Hauptmolekül hinzufügen (Kohlenstoffe als dicke Sphären, H klein)
    view.addModel(komplettes_mol, "mol")
    view.setStyle({'model': -1}, {
        'stick': {'colorscheme': 'Jmol', 'radius': 0.15},
        'sphere': {'colorscheme': 'Jmol', 'radius': 0.4}
    })

    # 4. Dummy-Atome für die Kugelwolken als transluzente Sphären injizieren
    for pos_w, typ in wolken_daten:
        # Farb-Mapping für das Lewis-Kugelwolkenmodell
        if typ == "bindung":
            farbe = "#00FFCC" # Cyan für kovalente Bindungswolken
            radius = 0.28
        elif typ == "paar":
            farbe = "#FFCC00" # Gelb für freie Elektronenpaare (Striche)
            radius = 0.25
        else:
            farbe = "#FF3333" # Rot für ungepaarte Radikale (Punkte)
            radius = 0.20

        view.addSphere({
            'center': {'x': float(pos_w[0]), 'y': float(pos_w[1]), 'z': float(pos_w[2])},
            'radius': radius,
            'color': farbe,
            'alpha': 0.75 # Leicht transparent, damit man die Geometrie durchsieht
        })

    view.zoomTo()
    return view.show()

# Aufruf des Plotters
visualisiere_kugelwolken_live()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Visualisierung von molekuel.mol

In [8]:
!pip install py3Dmol -q
import py3Dmol
import os

def zeige_3d_molekuel_lokal(dateiname="molekuel.mol"):
    # Prüfen, ob das Frontend die Datei bereits erzeugt hat
    if not os.path.exists(dateiname):
        print(f"Fehler: Die Datei '{dateiname}' wurde noch nicht exportiert.")
        print("Bitte gib zuerst im CLI den Befehl 'export' ein.")
        return

    try:
        # Die lokal gespeicherte .mol-Datei einlesen
        with open(dateiname, "r") as f:
            mol_data = f.read()

        if not mol_data.strip():
            print("Die exportierte Datei ist leer.")
            return

        # py3Dmol Viewer initialisieren
        view = py3Dmol.view(width=600, height=400)
        view.addModel(mol_data, "mol")

        # Schickes Rendering: Stick (Stäbchen) + Sphere (Kugeln für die Atome)
        view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.3}})

        # Automatisch auf das Molekül zentrieren und anzeigen
        view.zoomTo()
        view.show()

        print(f"🎉 '{dateiname}' erfolgreich in interaktives 3D-Modell geladen!")

    except Exception as e:
        print(f"Fehler beim Rendern der 3D-Grafik: {e}")

# Die Visualisierung starten
zeige_3d_molekuel_lokal()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

🎉 'molekuel.mol' erfolgreich in interaktives 3D-Modell geladen!


In [ ]:
# Nutzt deine bestehende py3Dmol-Funktion, um das eben erzeugte Molekül anzuzeigen
zeige_3d_molekuel_lokal("molekuel.mol")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

🎉 'molekuel.mol' erfolgreich in interaktives 3D-Modell geladen!
